## Physics-Informed Neural Network (PINN) 
Another possible way to solve the Black-Scholes model is through the use of machine learning. Partial Differential Equations (PDEs) are generally hard to solve and usually require numerical methods which can be computationally expensive. One approach currently being explored is a deep learning architecture known as a Physics-Informed Neural Network (PINN). This model is used to approximate the solutions to PDEs directly without a traditional mesh.

Given a general differential equation:

$$f(\vec{x}, t) = \frac{\partial u}{\partial t} + \mathcal{D}[u(\vec{x}, t)] = 0, \quad \vec{x} \in \Omega, \quad t \in [0, T]$$
 
where $\mathcal{D}$ is a non-linear differential operator. A neural network $\hat{u}(\vec{x},t; \theta)$ is trained to approximate the target function $u(\vec{x},t)$. The partial derivatives of $\hat{u}$ are calculated exactly using **Automatic Differentiation (AD)**. The total loss function for training the PINN is defined as:

$$MSE = MSE_u + MSE_f$$

$$MSE_u = \frac{1}{N_u} \sum_{i=1}^{N_u} \left| \hat{u}(\vec{x}_i^{(u)}, t_i^{(u)}; \theta) - u_i^{(u)} \right|^2$$

$$MSE_f = \frac{1}{N_f} \sum_{j=1}^{N_f} \left| f(\vec{x}_j^{(f)}, t_j^{(f)}; \theta) \right|^2$$

Where:
* $\hat{u}(\cdot; \theta)$ is the functional output of the neural network parameterized by weights and biases $\theta$.
* $(\vec{x}_i^{(u)}, t_i^{(u)})$ represent coordinates evaluated on the initial, terminal, or boundary constraints.
* $u_i^{(u)}$ represents the true specified value or target constraint at that point (e.g., terminal payoff).
* $f(\vec{x}_j^{(f)}, t_j^{(f)}; \theta)$ represents the residual evaluated at interior **collocation points**.

---

### References
* **Raissi et al. (2019):** *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving partial differential equations.* [Journal of Computational Physics](https://www.sciencedirect.com/science/article/abs/pii/S0021999118307125)
* **Louskos (2021):** *Physics-Informed Neural Networks for Pricing Financial Options.* [Dartmouth College Thesis](https://math.dartmouth.edu/theses/undergrad/2021/Louskos-thesis.pdf)
* **Makarov (2023):** *Applications of Physics-Informed Neural Networks in Financial Engineering.* [arXiv:2312.06711](https://arxiv.org/abs/2312.06711)


The model will be built using **PyTorch** and optimized using the built-in **Adam optimizer**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.cuda.init()

In [ ]:
# PINN model
# In features is S, t
class PINN_BS(nn.Module):

    def __init__(self, in_features = 2, out_features = 1, layers=3, hidden_layer=64):
        super().__init__()

        layer = [nn.Linear(in_features, hidden_layer), nn.Tanh()]
        for _ in range(layers):
            layer.append(nn.Linear(hidden_layer, hidden_layer))
            layer.append(nn.Tanh())
        layer.append(nn.Linear(hidden_layer, out_features))
        
        self.stack = nn.Sequential(*layer)
    
    def forward(self, x):
        return self.stack(x)

#### 2.1 Data and Boundary Loss ($MSE_u$)
The term $MSE_u$ enforces the boundary conditions ($S = 0$ and $S = S_{\max}$) as well as the terminal payoff condition ($t = T$). It is calculated as:

$$MSE_u = MSE_{bc1} + MSE_{bc2} + MSE_{term}$$

Where each component represents an average over its respective sampled boundary points:

* **Lower Boundary ($S = 0$):**
  $$MSE_{bc1} = \frac{1}{N_{b}} \sum_{i=1}^{N_{b}} \left| \hat{C}(0, t_i^{(b)}; \theta) - 0 \right|^2$$

* **Upper Boundary ($S = S_{\max}$):**
  $$MSE_{bc2} = \frac{1}{N_{b}} \sum_{i=1}^{N_{b}} \left| \hat{C}(S_{\max}, t_i^{(b)}; \theta) - \left( S_{\max} - K e^{-r(T - t_i^{(b)})} \right) \right|^2$$

* **Terminal Condition ($t = T$):**
  $$MSE_{term} = \frac{1}{N_{term}} \sum_{i=1}^{N_{term}} \left| \hat{C}(S_i^{(term)}, T; \theta) - \max(S_i^{(term)} - K, 0) \right|^2$$

Here, $\hat{C}$ denotes the option price predicted by the neural network, $N_{b}$ is the number of boundary evaluation points, and $N_{term}$ is the number of terminal samples.

---

#### 2.2 Residual Loss ($MSE_f$)
The term $MSE_f$ evaluates how well the network honors the underlying Black-Scholes PDE across $N_f$ collocation points randomly sampled from the interior domain $(S, t) \in (0, S_{\max}) \times (0, T)$. 

We define the interior physics residual $f(S, t)$ using automatic differentiation:

$$f(S, t) := \frac{\partial \hat{C}}{\partial t} + r S \frac{\partial \hat{C}}{\partial S} + \frac{1}{2} \sigma^2 S^2 \frac{\partial^2 \hat{C}}{\partial S^2} - r\hat{C}$$

The interior loss is the mean squared residual across all interior points:

$$MSE_f = \frac{1}{N_f} \sum_{j=1}^{N_f} \left| f(S_j^{(f)}, t_j^{(f)}; \theta) \right|^2$$

In [ ]:
S_max = 3 * K  # Upper Boundary for S -> inf

def collocation_points(S_max, T, n_pde=5000, n_boundary=500):
     # 1. Interior Domain Points (Randomly sampled PDE points)
    t_pde = (torch.rand(n_pde) * T).requires_grad_(True)
    s_pde = (torch.rand(n_pde) * S_max).requires_grad_(True)

    # 2. Lower Boundary (S = 0)
    t_low = torch.rand(n_boundary) * T
    s_low = torch.zeros(n_boundary)

    # 3. Upper Boundary (S = S_max)
    t_high = torch.rand(n_boundary) * T
    s_high = torch.ones(n_boundary) * S_max

    # 4. Terminal Payoff Boundary (t = T)
    t_term = torch.ones(n_boundary) * T
    s_term = torch.rand(n_boundary) * S_max

    s = [s_pde, s_low, s_high, s_term]
    t = [t_pde, t_low, t_high, t_term]

    return s, t

def plot_collocation():
    s, t = collocation_points(S_max, T)
    s_pde, s_low, s_high, s_term = s
    t_pde, t_low, t_high, t_term = t

    plt.figure(figsize=(10, 6))
    
    # 1. Interior Domain Points (Randomly sampled PDE points)
    plt.scatter(t_pde.detach().numpy(), s_pde.detach().numpy(), 
                color='royalblue', alpha=0.4, s=10, label='Interior PDE Points')

    # 2. Lower Boundary (S = 0)
    plt.scatter(t_low.numpy(), s_low.numpy(), 
                color='crimson', marker='o', s=15, label='Lower Bound: $C(0, t) = 0$')

    # 3. Upper Boundary (S = S_max)
    plt.scatter(t_high.numpy(), s_high.numpy(), 
                color='darkorange', marker='o', s=15, label='Upper Bound: $C(S_{max}, t)$')

    # 4. Terminal Payoff Boundary (t = T)
    plt.scatter(t_term.numpy(), s_term.numpy(), 
                color='forestgreen', marker='x', s=15, linewidths=2, label='Terminal Payoff: $t = T$')

    plt.title('PINN Collocation Points & Boundary Domains', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Time to Maturity ($t$)', fontsize=12)
    plt.ylabel('Stock Price ($S$)', fontsize=12)
    
    # Clean grid and neat layout margins
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xlim(-0.02, T + 0.05)
    plt.ylim(-5, S_max + 10)
    
    # Move legend outside or clear from points
    plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='gainsboro', fontsize=10)
    plt.tight_layout()
    plt.show()

plot_collocation()

In [ ]:
def compute_pinn_loss(s, t, model):
    r = 0.05       # Risk-free rate
    sigma = 0.20   # Volatility
    K = 100        # Strike price
    T = 1          # Time to maturity (1 year)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    s_pde, s_low, s_high, s_term = [x.to(device).unsqueeze(1) for x in s]
    t_pde, t_low, t_high, t_term = [x.to(device).unsqueeze(1) for x in t]
    
    # Lower Boundary Loss: V(0, t) = 0
    pred_low = model(torch.cat((s_low, t_low), dim=1))
    bc1_loss = F.mse_loss(pred_low, torch.zeros_like(pred_low))

    # Upper Boundary Loss: V(S_max, t) ≈ S_max - K * e^(-r*(T-t))
    pred_high = model(torch.cat((s_high, t_high), dim=1))
    target_high = s_high - K * torch.exp(-r * (T - t_high))
    bc2_loss = F.mse_loss(pred_high, target_high)

    # Terminal Payoff Loss: V(S, T) = max(S - K, 0)
    pred_term = model(torch.cat((s_term, t_term), dim=1))
    target_term = torch.clamp(s_term - K, min=0.0)
    terminal_loss = F.mse_loss(pred_term, target_term)

    # PDE Loss (Black-Scholes Residual)
    pred_pde = model(torch.cat((s_pde, t_pde), dim=1))
    
    dcs = torch.autograd.grad(pred_pde, s_pde, grad_outputs=torch.ones_like(pred_pde), create_graph=True)[0]
    d2cs = torch.autograd.grad(dcs, s_pde, grad_outputs=torch.ones_like(dcs), create_graph=True)[0]
    dct = torch.autograd.grad(pred_pde, t_pde, grad_outputs=torch.ones_like(pred_pde), create_graph=True)[0]

    pde_residual = r * s_pde * dcs + 1/2 * sigma ** 2 * s_pde ** 2 * d2cs - r * pred_pde + dct
    pde_loss = F.mse_loss(pde_residual, torch.zeros_like(pde_residual)) 
    
    return [bc1_loss, bc2_loss, terminal_loss, pde_loss]

In [ ]:
# Training PINN
def train_model(data, model, optimizer, epochs):

    losses = []
    for epoch in range(epochs + 1):
        optimizer.zero_grad()
        
        # Compute the custom loss
        s, t = data
        loss = compute_pinn_loss(s, t, model)
        total_loss_tensor = sum(loss)
        total_loss = total_loss_tensor.item()
        if epoch % 500 == 0:
            print(
                f"Epoch {epoch:<5d} | "
                f"Total Loss: {total_loss:>12.6f} | "
                f"BC1 Loss: {loss[0].item():>12.6f} | "
                f"BC2 Loss: {loss[1].item():>12.6f} | "
                f"Terminal Loss: {loss[2].item():>12.6f} | "
                f"ODE Loss: {loss[3].item():>12.6f}"
            )

        losses.append(total_loss)

        total_loss_tensor.backward()
        optimizer.step()

    return losses

model = PINN_BS()
epochs = 3500
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
data = collocation_points(S_max, T, n_pde=2000, n_boundary=500)
training_loss = train_model(data, model, optimizer, epochs)

In [ ]:
plt.plot(np.arange(0, epochs + 1), training_loss)
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.title("Training Loss")
plt.show()

In [ ]:
# Comparing PINN model solution with Monte Carlo and Analytical
def eval_model(S0_vals, time_vals, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    S0_vals = torch.from_numpy(S0_vals).float().unsqueeze(-1).to(device)
    t = torch.from_numpy(np.atleast_1d(time_vals)).float().to(device)
    t = t.repeat(S0_vals.shape[0], 1)
    x_ins = torch.cat((S0_vals, t), dim=-1)
    with torch.no_grad():
        y_pinn = model(x_ins) 
    return y_pinn.squeeze().detach().cpu().numpy()

def compare_methods(r, sigma, K, T, model):
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(13, 15))
    axs_flat = axs.flatten()
    time = np.linspace(0, 0.95, 9)
    S0_vals = np.linspace(5, 150, 50)
    mse_loss = 0

    for idx, t in enumerate(time):
        ax = axs_flat[idx]
        
        # Analytical
        bs_analytical = [BS(r, sigma, s, K, T) for s in S0_vals]

        # PINN
        bs_pinn = eval_model(S0_vals, t, model)

        # Analytical solution curve
        ax.plot(S0_vals, bs_analytical, label='Analytical Black-Scholes', linewidth=2)

        # PINN solution curve
        ax.plot(S0_vals, bs_pinn, label='PINN Black-Scholes')
        
        mse_loss += (bs_analytical - bs_pinn) ** 2
        # Graph formatting
        ax.set_title(f'Option Price at $t = {t:.2f}$', fontsize=10, fontweight='bold')
        ax.set_xlabel('Initial Stock Price ($S_0$)', fontsize=10)
        ax.set_ylabel('European Call Option Price', fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(fontsize=9)

    mse_loss = np.sum(mse_loss) / len(time)
    return mse_loss

In [ ]:
print(f"Total Loss: {compare_methods(r, sigma, K, T, model)}")

# 3. Optimization
One thing to notice is the PINN loss is large from the start of training, and even at the end the loss is still huge hence the estimation of black scholes is off as shown above. Some papers shown which optimizer is suitable for PINN. For optimizer tuning most uses Adam, LBFGS or combination of both Adam + LBFGS.
First order methods like SGD, AdaGrad are tested to see show it hardly can converge.
https://arxiv.org/abs/2501.16371
https://arxiv.org/abs/2402.01868
https://arxiv.org/abs/1912.07145

In [ ]:
from pyhessian import hessian
from pyhessian import get_esd_plot

In [ ]:
r = 0.05       # Risk-free rate
sigma = 0.20   # Volatility
K = 100        # Strike price
T = 1          # Time to maturity (1 year)
S_max = 3 * K  # Upper Boundary for S -> inf

In [ ]:
model = PINN_BS()
data = collocation_points(S_max, T, n_pde=2000, n_boundary=500)
h = hessian(model, compute_pinn_loss, data, device='cuda')
eigen_list_full, weight_list_full = h.density()
get_esd_plot(eigen_list_full, weight_list_full)

In [ ]:
def train_model(data, model, optimizer, epochs):
    
    loss_history = []
    current_losses = [0.0, 0.0, 0.0, 0.0]

    for epoch in range(epochs):
        # Closure function for L-BFGS
        def closure():
            optimizer.zero_grad()
            s, t = data
            loss_components = compute_pinn_loss(s, t, model)
            
            for i in range(4):
                current_losses[i] = loss_components[i].item()
            
            total_loss_tensor = sum(loss_components)
            total_loss_tensor.backward()
            return total_loss_tensor
        
        # Step the optimizer
        if isinstance(optimizer, torch.optim.LBFGS):
            total_loss = optimizer.step(closure)
            total_loss = sum(current_losses)
        else:
            optimizer.zero_grad()
            s, t = data
            loss_components = compute_pinn_loss(s, t, model)
            for i in range(4):
                current_losses[i] = loss_components[i].item()

            total_loss_tensor = sum(loss_components)
            total_loss_tensor.backward()
            optimizer.step()
            total_loss = total_loss_tensor.item()
            
        loss_history.append(total_loss)
        
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch + 1} | Total Loss: {total_loss:.6f}")
        
    return loss_history

In [ ]:
num_models = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models = [PINN_BS().to(device) for _ in range(num_models)]

configs = {
    "Adam": {
        "model": models[0],
        "optimizer": torch.optim.Adam(models[0].parameters(), lr=0.001),
        "epochs": 3000
    },
    "LBFGS_wolfe": {
        "model": models[1],
        "optimizer": torch.optim.LBFGS(models[1].parameters(), lr=1.0, max_iter=20, line_search_fn="strong_wolfe"),
        "epochs": 150
    },
    "AdaGrad": {
        "model": models[2],
        "optimizer": torch.optim.Adagrad(models[2].parameters(), lr=0.001),
        "epochs": 3000
    },
    "Adam+LBFGS" : {
        "model": models[3],
        "optimizer_LBFGS": torch.optim.LBFGS(models[3].parameters(), lr=1.0, max_iter=20, line_search_fn="strong_wolfe"),
        "optimizer_Adam" : torch.optim.Adam(models[3].parameters(), lr=0.001),
        "epochs_Adam": 1000,
        "epochs_LBFGS" : 100,
    }
}

In [ ]:
def optim_tune():
    results = {}
    data = collocation_points(S_max, T, n_pde=2000, n_boundary=500)
    for name, cfg in configs.items():
        print(f'Training with optimizer {name}')
        if name == "Adam+LBFGS":
            model = cfg["model"]
            epochs_Adam = cfg["epochs_Adam"]
            optimizer_Adam = cfg["optimizer_Adam"]
            
            loss_adam = train_model(data, model, optimizer_Adam, epochs_Adam)
            epochs_LBFGS = cfg["epochs_LBFGS"]
            optimizer_LBFGS = cfg["optimizer_LBFGS"]
            loss_LBFGS = train_model(data, model, optimizer_LBFGS, epochs_LBFGS)

            results[name] = {
                "x": np.concatenate((np.arange(0, epochs_Adam), np.arange(epochs_Adam, epochs_Adam + epochs_LBFGS * 2, 2))),
                "y": loss_adam + loss_LBFGS
            }
        else:
            epochs = cfg["epochs"]
            model = cfg["model"]
            optimizer = cfg["optimizer"]

            loss_y = train_model(data, model, optimizer, epochs)
            
            if name == "LBFGS_wolfe":
                results[name] = {
                    "x": np.arange(0, epochs * 6, 6),
                    "y": loss_y
                }
            else:
                results[name] = {
                    "x": np.arange(epochs),
                    "y": loss_y
                }
    return results

results = optim_tune()

In [ ]:
plt.figure(figsize=(10, 5))
for name, data in results.items():
    plt.plot(data["x"], data["y"], label=name, linewidth=2)
plt.xlabel('Epochs / Iterations')
plt.ylabel('Total Loss')
plt.title('PINN Optimizer Performance Comparison')
plt.legend()
plt.grid(True, which="both", ls="-")
plt.show()

In [ ]:
for name, data in results.items():
    print(f'Loss ({name}): {data["y"][-1]}')

In [ ]:
def weighted_loss(model, device):
    loss_components = compute_pinn_loss(model, device)
    weights = [100, 1, 1, 100]
    for i in range(len(loss_components)):
        loss_components[i] = loss_components[i] * weights[i]

    return loss_components

def train_model_weighted_loss(model, optimizer, epochs):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    losses = []
    for epoch in range(epochs + 1):
        optimizer.zero_grad()
        
        # Compute the custom loss
        loss = weighted_loss(model, device)
        total_loss_tensor = sum(loss)
        total_loss = total_loss_tensor.item()
        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Total Loss: {total_loss:.6f} | BC1 Loss: {loss[0].item():.6f} | BC2 Loss: {loss[1].item():.6f} | Terminal Loss: {loss[2].item():.6f} | ODE Loss: {loss[3].item():.6f}")

        losses.append(total_loss)

        total_loss_tensor.backward()
        optimizer.step()

    return losses

In [ ]:
model_weighted_loss = PINN_BS()
optimizer = torch.optim.Adam(model_weighted_loss.parameters(), lr=0.001)
epochs = 5000

loss = train_model_weighted_loss(model_weighted_loss, optimizer, epochs)
plt.plot(np.arange(epochs + 1), loss)
plt.title("Training Loss with weights")
plt.ylabel("Training Loss")
plt.xlabel("Epochs")


In [ ]:
print(f"Total Loss: {compare_methods(r, sigma, K, T, model_weighted_loss)}")